In [9]:
import time
import csv
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_total_params(model):
    return sum(p.numel() for p in model.parameters())

# def get_peak_memory():
#     return torch.cuda.max_memory_allocated() / 1024**2  # MB

os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

csv_file = "results/ResNet18_Lora_metrics.csv"


if not os.path.exists(csv_file):
    with open(csv_file, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "method", "epoch",
            "train_loss", "train_acc",
            "val_loss", "val_acc",
            "epoch_time_sec",
            "peak_memory_mb",
            "trainable_params"
        ])


# Pretrained EfficientNetV2-S weights
weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1



# Use simple preprocessing only
transform = weights.transforms()


# Load Food101
train_dataset_full = datasets.Food101(
    root="./data",
    split="train",
    download=True,
    transform=transform
)

test_dataset = datasets.Food101(
    root="./data",
    split="test",
    download=True,
    transform=transform
)

# Split training set into train + validation
train_size = int(0.9 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

Using device: mps


In [10]:
import torch.nn as nn
import torchvision.models as models
from torchvision.models import EfficientNet_V2_S_Weights
import loralib as lora

def apply_lora_outer_layers(model, rank=8, alpha=16, last_n_blocks=2):
    def to_int_if_square(x):
        if isinstance(x, tuple):
            assert x[0] == x[1], f"Non-square kernel not supported: {x}"
            return x[0]
        return x
    def replace_conv(module, parent, name):
        new_layer = lora.Conv2d(
            in_channels=module.in_channels,
            out_channels=module.out_channels,
            kernel_size=to_int_if_square(module.kernel_size),
            stride=to_int_if_square(module.stride),
            padding=to_int_if_square(module.padding),
            dilation=to_int_if_square(module.dilation),
            groups=module.groups,
            bias=(module.bias is not None),
            r=rank,
            lora_alpha=alpha,
        )

        # copy weights
        new_layer.conv.weight.data.copy_(module.weight.data)
        if module.bias is not None:
            new_layer.conv.bias.data.copy_(module.bias.data)

        setattr(parent, name, new_layer)

    def apply_to_block(block):
        for name, module in list(block.named_children()):
            if isinstance(module, nn.Conv2d) and module.groups == 1:
                replace_conv(module, block, name)
            else:
                apply_to_block(module)

    # --- 1. Apply to stem ---
    apply_to_block(model.features[0])

    # --- 2. Apply to last N blocks ---
    for block in model.features[-last_n_blocks:]:
        apply_to_block(block)
        
# Load pretrained model
model = models.efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)

# Replace classifier
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 101)

# Apply LoRA
apply_lora_outer_layers(model, rank=8, alpha=16, last_n_blocks=3)

# Freeze everything except LoRA
lora.mark_only_lora_as_trainable(model)

# Unfreeze classifier
for p in model.classifier.parameters():
    p.requires_grad = True



In [11]:
model = model.to(device)

In [12]:
for name, module in model.named_modules():
    if hasattr(module, "lora_A"):
        print(name, type(module))

features.0.0 <class 'loralib.layers.Conv2d'>
features.5.0.block.0.0 <class 'loralib.layers.Conv2d'>
features.5.0.block.2.fc1 <class 'loralib.layers.Conv2d'>
features.5.0.block.2.fc2 <class 'loralib.layers.Conv2d'>
features.5.0.block.3.0 <class 'loralib.layers.Conv2d'>
features.5.1.block.0.0 <class 'loralib.layers.Conv2d'>
features.5.1.block.2.fc1 <class 'loralib.layers.Conv2d'>
features.5.1.block.2.fc2 <class 'loralib.layers.Conv2d'>
features.5.1.block.3.0 <class 'loralib.layers.Conv2d'>
features.5.2.block.0.0 <class 'loralib.layers.Conv2d'>
features.5.2.block.2.fc1 <class 'loralib.layers.Conv2d'>
features.5.2.block.2.fc2 <class 'loralib.layers.Conv2d'>
features.5.2.block.3.0 <class 'loralib.layers.Conv2d'>
features.5.3.block.0.0 <class 'loralib.layers.Conv2d'>
features.5.3.block.2.fc1 <class 'loralib.layers.Conv2d'>
features.5.3.block.2.fc2 <class 'loralib.layers.Conv2d'>
features.5.3.block.3.0 <class 'loralib.layers.Conv2d'>
features.5.4.block.0.0 <class 'loralib.layers.Conv2d'>
feat

In [13]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

In [14]:

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    return avg_loss, acc

In [15]:
num_epochs = 8
best_val_acc = 0.0

trainable_params = count_trainable_params(model)
print(f"Trainable params: {trainable_params:,}")

for epoch in range(num_epochs):
    start_time = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    epoch_time = time.time() - start_time
    # peak_memory = get_peak_memory()

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
    print("-" * 40)

    # SAVE METRICS TO CSV
    with open(csv_file, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            'efficientnet_lora',
            epoch + 1,
            train_loss,
            train_acc,
            val_loss,
            val_acc,
            epoch_time,
            trainable_params
        ])

    # SAVE BEST MODEL
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save({
            "base_weights": "efficientnet_v2_s_imagenet1k_v1",  # identify backbone
            "model_state_dict": model.state_dict(),             # full fallback
            "lora_state_dict": lora.lora_state_dict(model),     # LoRA only
            "classifier_state_dict": model.classifier[1].state_dict(),  # FIXED
            "epoch": epoch,
            "val_acc": val_acc
        }, "models/lora_full_checkpoint_efficient.pth")

        print("Best LoRA checkpoint saved!")

Trainable params: 1,236,861
Epoch 1/8
Train Loss: 2.2202, Train Acc: 0.5015
Val Loss:   1.0958, Val Acc:   0.7189
----------------------------------------
Best LoRA checkpoint saved!
Epoch 2/8
Train Loss: 1.0913, Train Acc: 0.7194
Val Loss:   0.8345, Val Acc:   0.7747
----------------------------------------
Best LoRA checkpoint saved!
Epoch 3/8
Train Loss: 0.8734, Train Acc: 0.7694
Val Loss:   0.7369, Val Acc:   0.7984
----------------------------------------
Best LoRA checkpoint saved!
Epoch 4/8
Train Loss: 0.7505, Train Acc: 0.7978
Val Loss:   0.6741, Val Acc:   0.8147
----------------------------------------
Best LoRA checkpoint saved!
Epoch 5/8
Train Loss: 0.6653, Train Acc: 0.8199
Val Loss:   0.6337, Val Acc:   0.8281
----------------------------------------
Best LoRA checkpoint saved!
Epoch 6/8
Train Loss: 0.5954, Train Acc: 0.8374
Val Loss:   0.6228, Val Acc:   0.8313
----------------------------------------
Best LoRA checkpoint saved!
Epoch 7/8
Train Loss: 0.5384, Train Acc: 0